# Weekend Cineplex Scheduling Experiments

This notebook runs experiments for a **weekend** scheduling scenario across three dataset sizes:
- **Small**: 6 movies, 2 halls
- **Medium**: 12 movies, 6 halls
- **Large**: 30 movies, 15 halls

**Sections:**
1. Mutation rate tuning (small + medium)
2. Convergence plots (all datasets)
3. ILP vs GA scalability comparison
4. Solution quality vs runtime trade-off

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from heuristic_ga.ga_solver import solve_schedule_ga
from exact_method.exact_scheduler import solve_schedule_ilp

DAY_TYPE = "weekend"
SEEDS = [11, 22, 33, 44, 55]

DATASETS = {
    "small": {
        "movies_csv": REPO_ROOT / "common" / "movies_small.csv",
        "config_json": REPO_ROOT / "common" / "small_config.json",
    },
    "medium": {
        "movies_csv": REPO_ROOT / "common" / "movies_medium.csv",
        "config_json": REPO_ROOT / "common" / "med_config.json",
    },
    "large": {
        "movies_csv": REPO_ROOT / "common" / "movies_large.csv",
        "config_json": REPO_ROOT / "common" / "large_config.json",
    },
}

DATASET_NUM_MOVIES = {"small": 6, "medium": 12, "large": 30}

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 10,
})

print("Setup complete.")

## 1. Mutation Rate Tuning

Sweep `mutation_percent_genes` over `[5, 10, 15, 20, 25, 30, 40, 50]` on the **small** and **medium** datasets (5 seeds each). The ideal rate is selected by averaging the normalised mean revenue from both datasets so neither dominates.

In [ ]:
MUTATION_RATES = [5, 10, 15, 20, 25, 30, 40, 50]
TUNING_DATASETS = ["small", "medium"]

tuning_results: dict[str, dict[float, list[float]]] = {
    ds: {rate: [] for rate in MUTATION_RATES} for ds in TUNING_DATASETS
}

total_runs = len(TUNING_DATASETS) * len(MUTATION_RATES) * len(SEEDS)
run_i = 0

for ds_name in TUNING_DATASETS:
    ds = DATASETS[ds_name]
    for rate in MUTATION_RATES:
        for seed in SEEDS:
            run_i += 1
            print(f"\r[{run_i}/{total_runs}] {ds_name} | mut={rate}% | seed={seed}", end="", flush=True)
            result = solve_schedule_ga(
                movies_csv=ds["movies_csv"],
                config_json=ds["config_json"],
                day_type=DAY_TYPE,
                seed=seed,
                mutation_percent_genes=float(rate),
            )
            tuning_results[ds_name][rate].append(result["metadata"]["total_revenue"])

print("\nMutation-rate sweep complete.")

In [ ]:
mean_revenue: dict[str, list[float]] = {}
for ds_name in TUNING_DATASETS:
    mean_revenue[ds_name] = [
        np.mean(tuning_results[ds_name][rate]) for rate in MUTATION_RATES
    ]

normalised: dict[str, np.ndarray] = {}
for ds_name in TUNING_DATASETS:
    arr = np.array(mean_revenue[ds_name])
    normalised[ds_name] = arr / arr.max()

combined = np.mean([normalised[ds] for ds in TUNING_DATASETS], axis=0)

best_idx = int(np.argmax(combined))
IDEAL_MUTATION_RATE = MUTATION_RATES[best_idx]

print(f"Ideal mutation_percent_genes = {IDEAL_MUTATION_RATE}%")
print(f"  Combined normalised score  = {combined[best_idx]:.4f}")
for ds_name in TUNING_DATASETS:
    print(f"  {ds_name:>8s} normalised score = {normalised[ds_name][best_idx]:.4f}"
          f"  (mean revenue = {mean_revenue[ds_name][best_idx]:,.0f})")

# --- Plot ---
fig, ax = plt.subplots(figsize=(8, 4.5))

for ds_name in TUNING_DATASETS:
    ax.plot(MUTATION_RATES, normalised[ds_name], "o--", label=f"{ds_name.capitalize()}")

ax.plot(MUTATION_RATES, combined, "s-", color="black", linewidth=2, label="Combined")
ax.axvline(IDEAL_MUTATION_RATE, color="red", linestyle=":", alpha=0.6, label=f"Best = {IDEAL_MUTATION_RATE}%")

ax.set_xlabel("Mutation Rate (%)")
ax.set_ylabel("Normalised Mean Revenue")
ax.set_title("Mutation Rate Tuning (Weekend)")
ax.set_xticks(MUTATION_RATES)
ax.legend()
fig.tight_layout()
plt.show()

## 2. Convergence Plots

Run the GA on every dataset using the tuned mutation rate and plot **Best Fitness vs Generation** and **Average Fitness vs Generation** to check for premature convergence / local-minima traps.

In [ ]:
NUM_GENERATIONS = 300
CONVERGENCE_SEED = 42

convergence_data: dict[str, dict] = {}

for ds_name, ds in DATASETS.items():
    print(f"Running GA convergence for {ds_name}...", end=" ", flush=True)
    result = solve_schedule_ga(
        movies_csv=ds["movies_csv"],
        config_json=ds["config_json"],
        day_type=DAY_TYPE,
        seed=CONVERGENCE_SEED,
        num_generations=NUM_GENERATIONS,
        mutation_percent_genes=float(IDEAL_MUTATION_RATE),
    )
    convergence_data[ds_name] = result["convergence"]
    print(f"done  (revenue={result['metadata']['total_revenue']:,.0f},"
          f" time={result['metadata']['execution_time_seconds']:.1f}s)")

print("All convergence runs complete.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=False)

for ax, (ds_name, conv) in zip(axes, convergence_data.items()):
    gens = np.arange(1, len(conv["best_fitness_history"]) + 1)
    ax.plot(gens, conv["best_fitness_history"], label="Best Fitness", linewidth=1.4)
    ax.plot(gens, conv["avg_fitness_history"], label="Avg Fitness", linewidth=1.0, alpha=0.7)
    ax.set_xlabel("Generation")
    ax.set_ylabel("Fitness")
    ax.set_title(f"{ds_name.capitalize()} ({DATASET_NUM_MOVIES[ds_name]} movies)")
    ax.legend(fontsize=8)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

fig.suptitle(f"GA Convergence (mutation={IDEAL_MUTATION_RATE}%, weekend)", fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

## 3. ILP vs GA -- Scalability & Solution Quality

Run both solvers on every dataset to collect runtime and revenue for side-by-side comparison.

In [ ]:
ILP_TIME_LIMITS = {"small": 60, "medium": 180, "large": 600}

ilp_results: dict[str, dict] = {}

for ds_name, ds in DATASETS.items():
    tl = ILP_TIME_LIMITS[ds_name]
    print(f"Running ILP for {ds_name} (time_limit={tl}s)...", end=" ", flush=True)
    result = solve_schedule_ilp(
        movies_csv=ds["movies_csv"],
        config_json=ds["config_json"],
        day_type=DAY_TYPE,
        time_limit_seconds=tl,
    )
    meta = result["metadata"]
    ilp_results[ds_name] = {
        "revenue": meta["total_revenue"],
        "runtime": meta["execution_time_seconds"],
        "status": meta["solver_status"],
        "num_movies": meta["num_movies"],
    }
    print(f"done  status={meta['solver_status']}  revenue={meta['total_revenue']:,.0f}"
          f"  time={meta['execution_time_seconds']:.1f}s")

print("\nAll ILP runs complete.")

In [ ]:
ga_comparison_results: dict[str, dict] = {}

for ds_name, ds in DATASETS.items():
    print(f"Running GA for {ds_name} (mut={IDEAL_MUTATION_RATE}%)...", end=" ", flush=True)
    result = solve_schedule_ga(
        movies_csv=ds["movies_csv"],
        config_json=ds["config_json"],
        day_type=DAY_TYPE,
        seed=42,
        num_generations=NUM_GENERATIONS,
        mutation_percent_genes=float(IDEAL_MUTATION_RATE),
    )
    meta = result["metadata"]
    ga_comparison_results[ds_name] = {
        "revenue": meta["total_revenue"],
        "runtime": meta["execution_time_seconds"],
        "num_movies": meta["num_movies"],
    }
    print(f"done  revenue={meta['total_revenue']:,.0f}"
          f"  time={meta['execution_time_seconds']:.1f}s")

print("\nAll GA comparison runs complete.")

In [ ]:
ds_order = ["small", "medium", "large"]
num_movies = [DATASET_NUM_MOVIES[d] for d in ds_order]

ilp_runtimes = [ilp_results[d]["runtime"] for d in ds_order]
ga_runtimes = [ga_comparison_results[d]["runtime"] for d in ds_order]

fig, ax = plt.subplots(figsize=(7, 5))

ax.plot(num_movies, ilp_runtimes, "D-", markersize=8, linewidth=2, label="ILP (OR-Tools CBC)")
ax.plot(num_movies, ga_runtimes, "o-", markersize=8, linewidth=2, label="GA (PyGAD)")

for x, y_ilp, y_ga in zip(num_movies, ilp_runtimes, ga_runtimes):
    ax.annotate(f"{y_ilp:.1f}s", (x, y_ilp), textcoords="offset points",
                xytext=(0, 10), ha="center", fontsize=8, color="tab:blue")
    ax.annotate(f"{y_ga:.1f}s", (x, y_ga), textcoords="offset points",
                xytext=(0, -14), ha="center", fontsize=8, color="tab:orange")

ax.set_yscale("log")
ax.set_xlabel("Number of Movies (dataset size)")
ax.set_ylabel("Runtime (seconds, log scale)")
ax.set_title("Scalability: Runtime vs Dataset Size (Weekend)")
ax.set_xticks(num_movies)
ax.set_xticklabels([f"{n}\n({d.capitalize()})" for n, d in zip(num_movies, ds_order)])
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
revenue_ratio = []
speedup_factor = []
labels = []

for ds_name in ds_order:
    ilp_rev = ilp_results[ds_name]["revenue"]
    ga_rev = ga_comparison_results[ds_name]["revenue"]
    ilp_t = ilp_results[ds_name]["runtime"]
    ga_t = ga_comparison_results[ds_name]["runtime"]

    ratio = ga_rev / ilp_rev if ilp_rev > 0 else 0.0
    speedup = ilp_t / ga_t if ga_t > 0 else 0.0

    revenue_ratio.append(ratio)
    speedup_factor.append(speedup)
    labels.append(f"{ds_name.capitalize()}\n({DATASET_NUM_MOVIES[ds_name]} movies)")

x = np.arange(len(ds_order))
bar_w = 0.35

fig, ax1 = plt.subplots(figsize=(8, 5))
ax2 = ax1.twinx()

bars1 = ax1.bar(x - bar_w / 2, [r * 100 for r in revenue_ratio], bar_w,
                label="GA Revenue (% of ILP)", color="steelblue", zorder=3)
bars2 = ax2.bar(x + bar_w / 2, speedup_factor, bar_w,
                label="Speedup (ILP time / GA time)", color="darkorange", zorder=3)

for bar, val in zip(bars1, revenue_ratio):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
             f"{val * 100:.1f}%", ha="center", va="bottom", fontsize=9, color="steelblue")

for bar, val in zip(bars2, speedup_factor):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
             f"{val:.1f}x", ha="center", va="bottom", fontsize=9, color="darkorange")

ax1.set_xlabel("Dataset")
ax1.set_ylabel("GA Revenue as % of ILP Revenue", color="steelblue")
ax2.set_ylabel("Speedup Factor (ILP / GA)", color="darkorange")
ax1.set_xticks(x)
ax1.set_xticklabels(labels)
ax1.set_ylim(0, 110)
ax1.set_title("Solution Quality vs Runtime Trade-off (Weekend)")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=9)

fig.tight_layout()
plt.show()

In [ ]:
print("=" * 72)
print(f"{'Dataset':<10} {'ILP Rev':>12} {'GA Rev':>12} {'GA/ILP':>8} {'ILP Time':>10} {'GA Time':>10} {'Speedup':>8}")
print("-" * 72)
for ds_name in ds_order:
    ilp_rev = ilp_results[ds_name]["revenue"]
    ga_rev = ga_comparison_results[ds_name]["revenue"]
    ratio = ga_rev / ilp_rev if ilp_rev > 0 else 0.0
    ilp_t = ilp_results[ds_name]["runtime"]
    ga_t = ga_comparison_results[ds_name]["runtime"]
    speedup = ilp_t / ga_t if ga_t > 0 else 0.0
    print(f"{ds_name:<10} {ilp_rev:>12,.0f} {ga_rev:>12,.0f} {ratio:>7.1%} {ilp_t:>9.1f}s {ga_t:>9.1f}s {speedup:>7.1f}x")
print("=" * 72)

## 4. Summary & Observations

**Fill in after running all cells.**

### Mutation Rate Tuning
- Ideal `mutation_percent_genes` = __%
- Robustly selected by averaging normalised revenue from small and medium datasets across 5 seeds.

### Convergence Analysis
- **Small**: (describe plateau / continued improvement)
- **Medium**: (describe plateau / continued improvement)
- **Large**: (describe plateau / continued improvement)
- Does the GA appear to get stuck at local minima? (yes/no, and at which generation)
- Is further hyperparameter tuning warranted?

### Scalability
- ILP runtime grows steeply with dataset size (exponential); GA runtime scales near-linearly.
- On the large dataset, ILP took __s vs GA __s.

### Quality vs Speed Trade-off
- On small/medium the GA achieves __% / __% of ILP revenue.
- On large the GA achieves __% of ILP revenue while being __x faster.
- Conclusion: the GA provides a practical, scalable alternative when near-optimal solutions are acceptable.